In [ ]:
import re
import pandas as pd
from collections import Counter
import re


In [ ]:
path = '../data/small_samples/nosalary_sample_20k.parquet.gzip'

In [ ]:
usdf = pd.read_parquet(path)

In [ ]:
usdf.columns

In [ ]:
usdf['BODY']

In [ ]:


# Filter rows containing the word 'remote' in the 'JOB_DESCRIPTION' column if it is not NA
filtered_rows = usdf.dropna(subset=['BODY'])
filtered_rows = filtered_rows[filtered_rows['BODY'].str.contains(r'\bremote\b', flags=re.IGNORECASE, regex=True)]

# Extract the two words before and after 'remote' in each filtered row
phrases = []
for description in filtered_rows['BODY']:
    words = description.split()
    for i, word in enumerate(words):
        if word.lower() == 'remote':
            if i == 2:
                phrases.append(' '.join(words[i-2:i+4]))
            elif i >= 3:
                phrases.append(' '.join(words[i-3:i+4]))
            elif i == 1:
                phrases.append(' '.join(words[i-1:i+4]))
            elif i == 0:
                phrases.append(' '.join(words[i:i+5]))

# Count the occurrences of each phrase
phrase_counts = Counter(phrases)

# Get the top phrases
top_phrases = phrase_counts.most_common()

# Print the top phrases
for phrase, count in top_phrases:
    print(f'{phrase}: {count}')

In [ ]:

# list of keywords that suggest remote work
remote_keywords = ['remote', 'work from home', 'virtual position', 'remote position', 'telecommute', 'distributed team']

# Create a new column 'REMOTE_JOB' to flag remote jobs based on the presence of any keywords in 'JOB_DESCRIPTION'
# df['REMOTE_JOB'] = df['JOB_DESCRIPTION'].apply(lambda x: any(keyword in x.lower() for keyword in remote_keywords))

# get the count of remote and non-remote jobs by industry:
# remote_jobs_by_industry = df.groupby(['INDUSTRY', 'REMOTE_JOB']).size().reset_index(name='COUNT')


In [ ]:
# Define the regular expression pattern
pattern = r'((?:\w+\s+){{0,4}})({})((?:\s+\w+){{0,4}})'.format('|'.join(remote_keywords))

context_df = pd.DataFrame()

# Apply the regular expression pattern to 'JOB_DESCRIPTION' column and extract the matched strings
context_df['KEYWORD_CONTEXT'] = usdf['BODY'].apply(lambda x: re.findall(pattern, x.lower()))

# Extract the keyword and the word before and after it from the matched strings
context_df['KEYWORD_CONTEXT'] = context_df['KEYWORD_CONTEXT'].apply(lambda x: [' '.join(filter(None, item)).strip() for item in x])



In [ ]:

# Remove empty lists from 'KEYWORD_CONTEXT' column
context_df['KEYWORD_CONTEXT'] = context_df['KEYWORD_CONTEXT'].apply(lambda x: x if len(x) > 0 else None)
# Explode the 'KEYWORD_CONTEXT' column to have one phrase per row
context_df = context_df.explode('KEYWORD_CONTEXT')

In [ ]:
context_df[context_df['KEYWORD_CONTEXT'].notnull()]

# Label Remote Keywords

In [ ]:
remote_keywords = [
    "fully remote", "100% remote", "work from home", "remote role", "work remotely", 
    "remote eligible", "open to remote", "remote work environment", "remote work policy", 
    "remote and onsite", "virtual role", "remote first", "home office", "telecommute", 
    "distributed team", "remote-first", "remote-friendly", "remote options", "virtual work", "work from home", 
    "remote position", "wfh", "Remote - us", "telework", "home office", "(remote)", "remote work flexibility", "remote work: hybrid", 
    "remote: yes", "location: remote", "remote usa", "remote flexibility", "(remote", "- remote", "remote-flexibility", 
    "remote,", "\nremote\n", "\n remote \n", "remotely", "us-remote", "remote work eligible", "remote - united states", 
    "remote - work at home", "can be remote", "remote within the us", "remote in north america", "remote available" ", remote", 
    "remote full-time", "us remote", "#li-remote", "open to remote"
    
]
hybrid_keywords = [
    "hybrid work", "split time between office and remote", "office and remote options", 
    "partially remote", "hybrid position", "3 days remote", "hybrid workplace", 
    "some remote days", "in-office and remote", "hybrid onsite", "remote or in-office", "work-from-home days", "mix of working in the office and from home", "#li-hybrid", "hybrid remote"
]
exclusions = [
    "not considering remote", "in-office only", 
    "remote monitoring", "remote sensing", "remote access systems", "remote control", 
    "remote diagnostics", "remote delivery", "#li-onsite", "onsite job", "work from home not available", "telework:no", 
    "remote: no", "100% on-site", "work at home option: No",
    "remotely: no", "remote: n", "remotely: n", "telework: no", "remote: * no", "remotely: * no"
    "remotely piloted", "data remotely", "remote type on-site", "interviewed remotely", "work remotely: * no", "remotely piloted", 
    "remote desktop", "must be able to work on-site", "not applicable for 100% remote", "remote testing", "remotely upgrading", "remote usability",
    "remote site", "remote machine", "remote iot", "no remote", "work remotely no", "remotely:no", "remotely? n", "remotely no", "remote areas", 
    "remote access", "remote position? no", "remotely tucked away", "supporting remote", "remotely sensed"
]

#, "remote interview process"

In [ ]:
def clean_text(text):
    if isinstance(text, str):  # Ensure the value is a string
        text = text.strip()  # Remove leading and trailing spaces
        text = ' '.join(text.split())  # Replace multiple spaces with a single space
        text = text.replace('\n', ' ')  # Remove newline characters
    return text

# Apply the cleaning function to the Remote_KW column
usdf['BODY'] = usdf['BODY'].apply(clean_text)

In [ ]:
def label_remote_hybrid(row):
    body = row["BODY"] 
    if not body:  # Check if body is empty or None
        return 0
    if any(kw in body for kw in exclusions):
        return 0 
    elif any(kw in body for kw in remote_keywords):
        return 1    
    elif any(kw in body for kw in hybrid_keywords):
        return 1
    return 0

# Apply the labeling function
usdf["Remote_KW"] = usdf.apply(label_remote_hybrid, axis=1)

# Display the labeled DataFrame
usdf.head()

In [ ]:
usdf.Remote_KW.value_counts()

## WHAM Validation

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:


# Pivot the data to create a confusion matrix format
# cf_df = usdf[usdf['REMOTE_TYPE_NAME']!='[None]']
confusion_matrix = usdf.groupby(['Remote_KW', 'wfh_wham']).size().unstack().fillna(0)
confusion_matrix.columns = ['Not Remote/Hybrid (0)', 'Remote/Hybrid (1)']
confusion_matrix.index.name = 'Remote_KW'

# Plot the confusion matrix
plt.figure(figsize=(10, 6))
sns.heatmap(confusion_matrix, annot=True, fmt='g', cmap='Blues')
plt.ylabel('Remote Keywords')
plt.xlabel('WHAM Model')
plt.show()


In [ ]:
confusion_matrix

In [ ]:
confusion_matrix

In [ ]:
confusion_matrix

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
# Calculate metrics
y_true = usdf['Remote_KW']
y_pred = usdf['wfh_wham']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Print the metrics
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
for i, row in usdf[(usdf['Remote_KW'] == 0)&(usdf['wfh_wham']==1)].iterrows():
    print(row['BODY'])
    print('---')

In [ ]:
for i, row in usdf[(usdf['Remote_KW'] == 1)&(usdf['wfh_wham']==0)].iterrows():
    print(row['BODY'])
    print('---')

In [ ]:
usdf

# Even Sample

In [ ]:
even_sample = pd.read_parquet('../data/small_samples/salary_sample_2018_2023.parquet.gzip')

In [ ]:

# Apply the labeling function
even_sample["Remote_KW"] = even_sample.apply(label_remote_hybrid, axis=1)



In [ ]:
even_sample.groupby(['YEAR', 'Remote_KW']).size().reset_index(name='COUNT')

In [ ]:
even_sample.to_parquet('../data/small_samples/salary_sample_2018_2023.parquet.gzip', index=False)